<a href="https://colab.research.google.com/github/red-gunslinger/Exportar-un-modelo-de-ML/blob/main/Actividad_AES_Modos_Operacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pycryptodome pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 23.7 MB/s eta 0:00:00


In [2]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import json

In [3]:
def pad(data, block_size=16):
    pad_len = block_size - (len(data) % block_size)
    return data + bytes([pad_len]) * pad_len

def read_bmp(file_path):
    with open(file_path, "rb") as f:
        bmp = f.read()
    header = bmp[:54]
    body = bmp[54:]
    return header, body

def write_bmp(file_path, header, body):
    with open(file_path, "wb") as f:
        f.write(header + body)

In [4]:
def generate_crypto_material():
    material = {
        "key": get_random_bytes(16).hex(),
        "iv": get_random_bytes(16).hex(),
        "nonce": get_random_bytes(8).hex()
    }
    return material

def save_crypto_material(material, output_file):
    with open(output_file, "w") as f:
        json.dump(material, f, indent=4)

def load_crypto_material(input_file):
    with open(input_file, "r") as f:
        material = json.load(f)
    return material

In [5]:
material = generate_crypto_material()
save_crypto_material(material, "mis_claves_MATRICULA.json")
print(material)

{'key': 'eef1154af760ac91a431d160e9fac796', 'iv': '01fdcfbc5ae307402c331afdca56c046', 'nonce': '2e67929766f93698'}


In [6]:
def encrypt_bmp(input_file, output_file, mode_name, material):
    header, body = read_bmp(input_file)
    key = bytes.fromhex(material["key"])

    if mode_name == "ECB":
        cipher = AES.new(key, AES.MODE_ECB)
        body_padded = pad(body)
        encrypted = cipher.encrypt(body_padded)
        encrypted = encrypted[:len(body)]

    elif mode_name == "CBC":
        iv = bytes.fromhex(material["iv"])
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        body_padded = pad(body)
        encrypted = cipher.encrypt(body_padded)
        encrypted = encrypted[:len(body)]

    elif mode_name == "CTR":
        nonce = bytes.fromhex(material["nonce"])
        cipher = AES.new(key, AES.MODE_CTR, nonce=nonce)
        encrypted = cipher.encrypt(body)

    else:
        raise ValueError("Modo no soportado")

    write_bmp(output_file, header, encrypted)
    print(f"{mode_name} cifrado guardado en: {output_file}")

In [7]:
material = load_crypto_material("mis_claves_MATRICULA.json")

encrypt_bmp("input.bmp", "ecb.bmp", "ECB", material)
encrypt_bmp("input.bmp", "cbc.bmp", "CBC", material)
encrypt_bmp("input.bmp", "ctr.bmp", "CTR", material)

ECB cifrado guardado en: ecb.bmp
CBC cifrado guardado en: cbc.bmp
CTR cifrado guardado en: ctr.bmp


In [8]:
def decrypt_bmp(input_file, output_file, mode_name, material):
    header, body = read_bmp(input_file)
    key = bytes.fromhex(material["key"])

    if mode_name == "ECB":
        cipher = AES.new(key, AES.MODE_ECB)
        decrypted = cipher.decrypt(pad(body))
        decrypted = decrypted[:len(body)]

    elif mode_name == "CBC":
        iv = bytes.fromhex(material["iv"])
        cipher = AES.new(key, AES.MODE_CBC, iv=iv)
        decrypted = cipher.decrypt(pad(body))
        decrypted = decrypted[:len(body)]

    elif mode_name == "CTR":
        nonce = bytes.fromhex(material["nonce"])
        cipher = AES.new(key, AES.MODE_CTR, nonce=nonce)
        decrypted = cipher.decrypt(body)

    else:
        raise ValueError("Modo no soportado")

    write_bmp(output_file, header, decrypted)
    print(f"{mode_name} descifrado guardado en: {output_file}")

In [9]:
material = load_crypto_material("mis_claves_MATRICULA.json")

decrypt_bmp("ecb.bmp", "ecb_descifrada.bmp", "ECB", material)
decrypt_bmp("cbc.bmp", "cbc_descifrada.bmp", "CBC", material)
decrypt_bmp("ctr.bmp", "ctr_descifrada.bmp", "CTR", material)

ECB descifrado guardado en: ecb_descifrada.bmp
CBC descifrado guardado en: cbc_descifrada.bmp
CTR descifrado guardado en: ctr_descifrada.bmp


In [10]:
def flip_byte(file_path, output_path, position):
    with open(file_path, "rb") as f:
        data = bytearray(f.read())

    print("Byte original:", data[position])
    data[position] ^= 0xFF
    print("Byte modificado:", data[position])

    with open(output_path, "wb") as f:
        f.write(data)

    print("Archivo modificado guardado en:", output_path)

In [11]:
flip_byte("ecb.bmp", "ecb_mod.bmp", 2000)
flip_byte("cbc.bmp", "cbc_mod.bmp", 2000)
flip_byte("ctr.bmp", "ctr_mod.bmp", 2000)

Byte original: 130
Byte modificado: 125
Archivo modificado guardado en: ecb_mod.bmp
Byte original: 53
Byte modificado: 202
Archivo modificado guardado en: cbc_mod.bmp
Byte original: 23
Byte modificado: 232
Archivo modificado guardado en: ctr_mod.bmp


In [12]:
material = load_crypto_material("mis_claves_MATRICULA.json")

decrypt_bmp("ecb_mod.bmp", "ecb_mod_descifrada.bmp", "ECB", material)
decrypt_bmp("cbc_mod.bmp", "cbc_mod_descifrada.bmp", "CBC", material)
decrypt_bmp("ctr_mod.bmp", "ctr_mod_descifrada.bmp", "CTR", material)

ECB descifrado guardado en: ecb_mod_descifrada.bmp
CBC descifrado guardado en: cbc_mod_descifrada.bmp
CTR descifrado guardado en: ctr_mod_descifrada.bmp


COMPANERO

In [13]:
material_companero = load_crypto_material("claves_companero.json")
decrypt_bmp(
    "imagen_companero.bmp",
    "imagen_companero_descifrada.bmp",
    "CBC",
    material_companero
)

CBC descifrado guardado en: imagen_companero_descifrada.bmp


In [14]:
mis_materiales = load_crypto_material("mis_claves_MATRICULA.json")
decrypt_bmp(
    "imagen_companero.bmp",
    "fallo_descifrado.bmp",
    "CBC",
    mis_materiales
)

CBC descifrado guardado en: fallo_descifrado.bmp


11.
Cuando se uso la llave correcta de carlos con los parametros correctos se pudo descifrar la imagen sin problema. La imagen se vio bien y fue posible reconocer su contenido. Esto muestra que el proceso funciona correctamente cuando solo se usa la informacion correcta.

12.

Cuando se uso una llave incorrecta la imagen no se pudo recuperar de forma correcta. El archivo se genero pero la imagen se veia dañada o super rara. Esto demuestra que aunque el proceso se ejecute la informacion no es valida si la llave no es la correcta.

13.

Este experimento demuestra que la llave es lo mas importante en el proceso de cifrado y descifrado. Sin la llave correcta no es posible recuperar la informacion original. Aunque se conozca el metodo utilizado la llave es necesaria para obtener un resultado valido.

En esta actividad se pudo ver que la diferencia entre ECB CBC y CTR es la forma en que manejan la informacion. ECB deja ver patrones de la imagen original y que no es seguro para informacion con estructura. CBC y CTR ocultan mejor la informacion y generan salidas que parecen mas random. Tambien se aprendio que la llave es esencial ya que solo con la llave correcta se pudo recuperar la imagen original. Cuando se uso una llave mala no se obtuvo un resultado valido. Por ultimo, el experimento de modificar un byte enseno que cada modo reacciona de forma distinta ante errores ya que algunos afectan mas partes de la imagen y otros solo una zona pequeña.